In [ ]:
import requests
import time
from IPython.display import HTML, display, clear_output

API = "http://localhost:5001"

def fmt(n):
    if n >= 1_000_000: return f"{n/1_000_000:.1f} M zl"
    if n >= 1000: return f"{n/1000:.0f} k zl"
    return f"{n} zl"

def render(data):
    cats = data['top_categories']
    prods = data['top_products']
    max_rev = cats[0]['total_revenue'] if cats else 1
    max_ord = prods[0]['order_count'] if prods else 1
    def bars(items, vkey, nkey, color, suffix):
        rows = ''
        mx = items[0][vkey] if items else 1
        for r in items:
            v = r[vkey]
            pct = round(v/mx*100) if mx else 0
            label = fmt(v) if suffix=='zl' else f'{v} zam.'
            rows += f'''<div style="margin-bottom:10px">
              <div style="display:flex;justify-content:space-between;font-size:13px;margin-bottom:4px">
                <span><span style="color:#64748b;font-size:11px">#{r['rank']} </span>{r[nkey]}</span>
                <span style="color:#64748b">{label}</span></div>
              <div style="background:#2d3148;border-radius:4px;height:8px">
                <div style="height:8px;border-radius:4px;width:{pct}%;background:{color}"></div></div></div>'''
        return rows
    html = f'''<div style="font-family:system-ui,sans-serif;background:#0f1117;color:#e2e8f0;padding:24px;border-radius:12px">
      <h1 style="font-size:22px;font-weight:500;color:#f8fafc;margin:0 0 8px">RTA Dashboard</h1>
      <p style="font-size:13px;color:#64748b;margin:0 0 24px">Aktualizacja: {data['timestamp'][11:19]} UTC</p>
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px">
        <div style="background:#1e2130;border-radius:12px;padding:20px;border:1px solid #2d3148">
          <h2 style="font-size:14px;font-weight:500;color:#94a3b8;margin:0 0 16px;text-transform:uppercase">Top kategorie - przychod (1 min)</h2>
          {bars(cats,'total_revenue','category','linear-gradient(90deg,#6366f1,#8b5cf6)','zl')}</div>
        <div style="background:#1e2130;border-radius:12px;padding:20px;border:1px solid #2d3148">
          <h2 style="font-size:14px;font-weight:500;color:#94a3b8;margin:0 0 16px;text-transform:uppercase">Top produkty - zamowienia (5 min)</h2>
          {bars(prods,'order_count','product_name','linear-gradient(90deg,#10b981,#34d399)','zam')}</div>
      </div></div>'''
    return html

# Petla auto-odswiezania (zatrzymaj kwadratem stop)
try:
    while True:
        r = requests.get(f"{API}/summary?n=7&m=10", timeout=5)
        data = r.json()
        clear_output(wait=True)
        display(HTML(render(data)))
        time.sleep(10)
except KeyboardInterrupt:
    print("Zatrzymano")